In [0]:
dbutils.widgets.text("event_blob_path", "")

event_blob_path = dbutils.widgets.get("event_blob_path")

if not event_blob_path:
    raise ValueError(
        "event_blob_path is empty. This notebook must be executed from ADF."
    )

print("Event blob path:")
print(event_blob_path)

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-4556313941473040>, line 6
      3 event_blob_path = dbutils.widgets.get("event_blob_path")
      5 if not event_blob_path:
----> 6     raise ValueError(
      7         "event_blob_path is empty. This notebook must be executed from ADF."
      8     )
     10 print("Event blob path:")
     11 print(event_blob_path)

ValueError: event_blob_path is empty. This notebook must be executed from ADF.

In [0]:
relative_path = event_blob_path.replace("bronze/", "", 1)

bronze_path = (
    "abfss://bronze@stcommercesrc.dfs.core.windows.net/"
    + relative_path
)

print("Bronze path:")
print(bronze_path)

Bronze path:
abfss://bronze@stcommercesrc.dfs.core.windows.net/


In [0]:
print("STEP 1 - Reading Bronze order file")

df_bronze = spark.read.parquet(bronze_path)

rows_read = df_bronze.count()

print("Rows read from Bronze:", rows_read)

df_bronze.printSchema()

display(df_bronze.limit(10))

STEP 1 - Reading Bronze product file


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4556313941473043>, line 5
      1 print("STEP 1 - Reading Bronze product file")
      3 df_bronze = spark.read.parquet(bronze_path)
----> 5 rows_read = df_bronze.count()
      7 print("Rows read from Bronze:", rows_read)
      9 display(df_bronze.limit(10))

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:1923, in DataFrame._to_table(self)
   1921 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1922     query = self._plan.to_proto(self._session.client)
-> 1923     table, schema, self._execut

In [0]:
print("STEP 2 - Inspecting Orders schema")

df_bronze.printSchema()

print("Total Bronze rows:", df_bronze.count())

print("Sample Orders data:")
display(df_bronze.limit(10))

In [0]:
from pyspark.sql.functions import col, trim

if "order_status" in df_bronze.columns:
    display(
        df_bronze
        .groupBy("order_status")
        .count()
        .orderBy("order_status")
    )
else:
    print("order_status column not found. Check the schema above.")

In [0]:
from pyspark.sql.functions import col, trim, upper, greatest

print("STEP 5.1 - Cleaning Orders data")

df_clean = (
    df_bronze
    # Standardize status columns
    .withColumn("order_status", upper(trim(col("order_status"))))
    .withColumn("payment_status", upper(trim(col("payment_status"))))
    .withColumn("shipping_status", upper(trim(col("shipping_status"))))

    # Standardize monetary columns
    .withColumn("total_amount", col("total_amount").cast("decimal(12,2)"))
    .withColumn("discount_amount", col("discount_amount").cast("decimal(12,2)"))
    .withColumn("tax_amount", col("tax_amount").cast("decimal(12,2)"))
    .withColumn("final_amount", col("final_amount").cast("decimal(12,2)"))
)

print("Bronze rows:", df_bronze.count())
print("Rows after cleaning:", df_clean.count())

display(df_clean.limit(10))

In [0]:
print("Order statuses after cleaning:")

display(
    df_clean
    .groupBy("order_status")
    .count()
    .orderBy("order_status")
)

print("Null checks:")

display(
    df_clean.select(
        *[
            col(c).isNull().cast("int").alias(c)
            for c in [
                "order_id",
                "customer_id",
                "order_date",
                "order_status",
                "total_amount",
                "final_amount"
            ]
        ]
    )
    .groupBy()
    .sum()
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

print("STEP 6.1 - Deduplicating Orders")

window_spec = (
    Window
    .partitionBy("order_id")
    .orderBy(
        col("updated_at").desc_nulls_last(),
        col("created_at").desc_nulls_last()
    )
)

df_ranked = df_clean.withColumn(
    "row_num",
    row_number().over(window_spec)
)

df_deduplicated = (
    df_ranked
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("Rows before deduplication:", df_clean.count())
print("Rows after deduplication:", df_deduplicated.count())
print(
    "Duplicate rows removed:",
    df_clean.count() - df_deduplicated.count()
)

In [0]:
from pyspark.sql.functions import upper, trim

print("STEP 6.2 - Validating Orders")

valid_order_statuses = [
    "PLACED",
    "CONFIRMED",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED"
]

df_valid = df_deduplicated.filter(
    col("order_id").isNotNull()
    & col("customer_id").isNotNull()
    & col("order_date").isNotNull()
    & col("order_status").isin(valid_order_statuses)
    & col("total_amount").isNotNull()
    & (col("total_amount") >= 0)
    & col("discount_amount").isNotNull()
    & (col("discount_amount") >= 0)
    & col("tax_amount").isNotNull()
    & (col("tax_amount") >= 0)
    & col("final_amount").isNotNull()
    & (col("final_amount") >= 0)
)

df_rejected = df_deduplicated.join(
    df_valid.select("order_id"),
    on="order_id",
    how="left_anti"
)

print("Deduplicated rows:", df_deduplicated.count())
print("Valid Orders:", df_valid.count())
print("Rejected Orders:", df_rejected.count())

display(df_rejected.limit(10))

In [0]:
print("STEP 6.3 - Orders quality summary")

display(
    df_valid.groupBy("order_status")
    .count()
    .orderBy("order_status")
)

print("Valid order IDs:", df_valid.select("order_id").distinct().count())

In [0]:
from delta.tables import DeltaTable

print("STEP 7.2 - Merging Orders into Silver")

target_table = "ecommerce.silver.orders_external"

if not spark.catalog.tableExists(target_table):
    raise ValueError(f"Target table does not exist: {target_table}")

target = DeltaTable.forName(spark, target_table)

(
    target.alias("t")
    .merge(
        df_valid.alias("s"),
        "t.order_id = s.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Orders MERGE completed successfully.")

In [0]:
%sql
SELECT COUNT(*) FROM ecommerce.silver.orders_external
WHERE payment_status = "PAID"

count(1)
75057
